|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 4:</h2>|<h1>The Scheduler<h1>|
|<h2>Section:</h2>|<h1>Admission and preemption<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: the scheduler that does not fall over<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

Write the scheduler that does not fall over.

Stage 05's scheduler assumed memory was infinite. This one has a block budget
that a running sequence can exhaust at any step, and it has to stay correct
when that happens.

All simulation. The GPU is a counter, which is the right way to get this
right before it is fast.

In [ ]:
### run this cell

BLOCK   = 16
PROMPT  = 48
lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=5000).astype(int) + 1

from dataclasses import dataclass

@dataclass(eq=False)
class Sequence:
  length: int        # tokens that it holds now: the prompt, then each new token
  target: int        # the length at which it finishes

def blocks_for(num_tokens):
  return -(-num_tokens // BLOCK)                # ceiling division

print(f'median sequence: {blocks_for(int(np.median(lengths)))} blocks, p99: {blocks_for(int(np.percentile(lengths,99)))} blocks')

# Exercise 1: three queues and a budget

Four methods: `arrive`, `admit`, `preempt`, `step`.

The interesting one is `step`. A sequence crossing into a new block may find
the pool empty, and the answer is not to fail. It is to take the blocks back
from somebody else.

In [ ]:
class Scheduler:
  def __init__(self, pool_blocks, watermark=0.05, max_running=64):
    self.free = pool_blocks
    self.reserve = max(1, int(watermark*pool_blocks))
    self.max_running = max_running          # max_num_seqs
    self.waiting, self.running = [], []
    self.preemptions = self.recomputed = self.work = 0

  def arrive(self, total_len):
    self.waiting.append(Sequence(PROMPT, total_len))

  def admit(self):
    while (self.waiting and len(self.running) < self.max_running
           and self.free - blocks_for(self.waiting[0].length) >= self.reserve):
      seq = self.waiting.pop(0)
      self.free -= blocks_for(seq.length)
      self.running.append(seq)

  def preempt(self, protect):
    victim = self.running[-1] if self.running[-1] is not protect else self.running[-2]
    self.free += blocks_for(victim.length)
    self.recomputed += victim.length - PROMPT
    victim.length = PROMPT
    self.running.remove(victim)
    self.waiting.insert(0, victim)
    self.preemptions += 1

  def step(self):
    self.admit()
    for seq in list(self.running):
      if seq not in self.running:
        continue
      if seq.length % BLOCK == 0:
        while self.free == 0 and len(self.running) > 1:
          self.preempt(seq)
        if self.free == 0:
          return
        self.free -= 1
      seq.length += 1
      self.work += 1               # a token the GPU actually produced
      if seq.length >= seq.target:
        self.running.remove(seq)
        self.free += blocks_for(seq.length)

scheduler = Scheduler(2000)
for index in range(400):
  scheduler.arrive(PROMPT + int(lengths[index]))
steps = 0
while (scheduler.waiting or scheduler.running) and steps < 200000:
  scheduler.step()
  steps += 1
print(f'{steps:,} steps, {scheduler.preemptions} preemptions, '
      f'{100*scheduler.recomputed/scheduler.work:.1f}% of the work was done twice')

# Exercise 2: how hard can you squeeze?

Sweep the pool size. Watch for where degradation stops being graceful.

In [ ]:
def run(pool, watermark=0.05, num_requests=400):
  scheduler = Scheduler(pool, watermark)
  for index in range(num_requests):
    scheduler.arrive(PROMPT + int(lengths[index]))
  steps = 0
  while (scheduler.waiting or scheduler.running) and steps < 200000:
    scheduler.step()
    steps += 1
  return steps, scheduler.preemptions, scheduler.recomputed, scheduler.work

pools = [300, 500, 750, 1000, 1500, 2000, 3000, 4000]
rows = [run(pool) for pool in pools]
roomy_steps = rows[-1][0]

print(f"{'pool':>6} {'steps':>8} {'vs roomy':>9} {'preempt':>8} {'wasted work':>12}")
for pool, (steps, preemptions, recomputed, work) in zip(pools, rows):
  print(f'{pool:>6} {steps:>8,} {steps/roomy_steps:>8.2f}x {preemptions:>8} '
        f'{100*recomputed/work:>11.1f}%')

# Exercise 3: the watermark

The reserve is the only thing stopping a new arrival from starving everyone
already running. Try turning it off.

In [ ]:
print(f"{'watermark':>10} {'steps':>8} {'preempt':>8} {'wasted work':>12}")
for watermark in (0.0, 0.01, 0.05, 0.15, 0.30, 0.50):
  steps, preemptions, recomputed, work = run(600, watermark=watermark)
  print(f'{watermark:>10.0%} {steps:>8,} {preemptions:>8} {100*recomputed/work:>11.1f}%')

### What the two sweeps say

**A smaller pool costs nothing, until it costs a lot.** The steps column is
flat across most of the range. It then turns up sharply at the small end,
where the repeated work climbs above 10%.

That flat region is the reason to build PagedAttention. You can run much
closer to the edge than a contiguous allocator permits. You pay nothing for
it, up to the knee.

**The watermark buys correctness, and the price is an idle machine.** At 0%
the scheduler is greedy. It gives the most preemptions and the most repeated
work, and it is still near the best on total steps. A preempted sequence frees
a slot that another sequence uses at once.

Raise the watermark, and the preemptions go to zero. Raise it more, and the
step count climbs, because the reserve is memory that you choose not to use.

So this is a real trade, and not a defect to correct. You pay repeated work on
one side and an idle machine on the other. The useful values are small and
above zero. Such a value is hard to guess and easy to measure. You set this
constant once from a sweep like this one, and then you leave it alone for
years.

**And look at which sequence you preempt.** You preempt the newest one. This
choice is not arbitrary. The newest sequence made the fewest tokens, so it is
the cheapest one to compute again. The request that waited longest keeps its
work. If you preempt the oldest sequence, you throw away the most work.

    ./vc guide 10